# Task 3: Multimodal ML — Housing Price Prediction Using Images + Tabular Data

**DevelopersHub Corporation — AI/ML Engineering Internship (Advanced Task 3)**

## Problem Statement & Objective
Real estate prices depend not only on structured attributes (bedrooms, bathrooms, square footage, location)
but also on visual factors (kerb appeal, interior finish, natural light) that are only visible in photos.
The objective of this notebook is to build a **multimodal regression model** that predicts a house's sale
price by fusing:

1. **Tabular features** — number of bedrooms, bathrooms, square footage, and zip code.
2. **Image features** — extracted with a Convolutional Neural Network (CNN) from photos of the house
   (bedroom, bathroom, kitchen, and frontal/exterior view).

We combine both feature streams (**feature fusion**) into a single network and train it end-to-end to
predict price, then evaluate with **MAE** and **RMSE**.

## Dataset
We use the public **Houses Dataset** (Ahmed & Moustafa, 2016), which was purpose-built for this exact
image + tabular housing-price task. It contains **535 houses**, each with:
- 4 images per house (bathroom, bedroom, kitchen, frontal exterior)
- A text file (`HousesInfo.txt`) with: `bedrooms bathrooms area zipcode price`

Source: https://github.com/emanhamed/Houses-dataset


## 1. Environment Setup
Run this in **Google Colab** with a GPU runtime: `Runtime > Change runtime type > GPU`.

In [ ]:
# Install/confirm required libraries (Colab already has most of these)
!pip install -q tensorflow scikit-learn matplotlib pandas seaborn


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Dataset Loading

We clone the dataset directly from GitHub. It contains the tabular info file plus 4 JPEGs per house
named `{house_id}_bathroom.jpg`, `{house_id}_bedroom.jpg`, `{house_id}_frontal.jpg`, `{house_id}_kitchen.jpg`.


In [ ]:
!git clone --depth 1 https://github.com/emanhamed/Houses-dataset.git
DATA_DIR = "Houses-dataset/Houses Dataset"
print(os.listdir(DATA_DIR)[:10])


In [ ]:
# Load tabular data. The info file has no header: bedrooms bathrooms area zipcode price
cols = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
df = pd.read_csv(os.path.join(DATA_DIR, "HousesInfo.txt"), sep=" ", header=None, names=cols)
df["house_id"] = df.index + 1  # dataset is 1-indexed and images are named accordingly
print(df.shape)
df.head()


## 3. Preprocessing

### 3.1 Tabular preprocessing
- `zipcode` is high-cardinality (many unique values with few samples each), so instead of one-hot
  encoding (which would explode dimensionality on only 535 rows) we use **target encoding**: each
  zipcode is replaced by the *mean training-set house price* for that zipcode. This is fit only on the
  training split to avoid data leakage.
- `bedrooms`, `bathrooms`, `area`, and the encoded zipcode are scaled to [0, 1] with `MinMaxScaler`.
- `price` (the target) is scaled to [0, 1] as well for stable training, then inverted back to dollars
  for evaluation.

### 3.2 Image preprocessing
Each house has 4 separate photos. Following the standard approach for this dataset, we combine the
4 images into a single **2x2 montage** (bathroom, bedroom top row; frontal, kitchen bottom row) so the
CNN sees the whole house in one image. The montage is resized to 224x224 (MobileNetV2's expected input
size).


In [ ]:
IMG_SIZE = 224  # required input size for MobileNetV2

def load_house_montage(house_id, data_dir=DATA_DIR, size=IMG_SIZE):
    room_types = ["bathroom", "bedroom", "frontal", "kitchen"]
    images = []
    for room in room_types:
        path = os.path.join(data_dir, f"{house_id}_{room}.jpg")
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (size // 2, size // 2))
        images.append(img)
    top = np.hstack([images[0], images[1]])     # bathroom | bedroom
    bottom = np.hstack([images[2], images[3]])  # frontal  | kitchen
    montage = np.vstack([top, bottom])          # final size x size, 4 rooms visible
    return montage

# Sanity check: visualize one montage
sample_montage = load_house_montage(df["house_id"].iloc[0])
plt.figure(figsize=(4, 4))
plt.imshow(sample_montage)
plt.title(f"House #{df['house_id'].iloc[0]} montage (bathroom | bedroom / frontal | kitchen)")
plt.axis("off")
plt.show()


In [ ]:
# Load all montages into memory (535 small images fits comfortably in Colab RAM)
images = np.array([load_house_montage(hid) for hid in df["house_id"]], dtype="float32")
print("Image tensor shape:", images.shape)


In [ ]:
# Train/test split BEFORE any target encoding / scaling to avoid data leakage
train_df, test_df, train_images, test_images = train_test_split(
    df, images, test_size=0.2, random_state=SEED
)

# --- Target-encode zipcode using ONLY training data ---
zip_price_map = train_df.groupby("zipcode")["price"].mean()
global_mean_price = train_df["price"].mean()

train_df = train_df.copy()
test_df = test_df.copy()
train_df["zipcode_encoded"] = train_df["zipcode"].map(zip_price_map)
test_df["zipcode_encoded"] = test_df["zipcode"].map(zip_price_map).fillna(global_mean_price)

tabular_cols = ["bedrooms", "bathrooms", "area", "zipcode_encoded"]

tab_scaler = MinMaxScaler()
X_train_tab = tab_scaler.fit_transform(train_df[tabular_cols])
X_test_tab = tab_scaler.transform(test_df[tabular_cols])

price_scaler = MinMaxScaler()
y_train = price_scaler.fit_transform(train_df[["price"]])
y_test = price_scaler.transform(test_df[["price"]])

# Image preprocessing expected by MobileNetV2 (scales pixels to [-1, 1])
X_train_img = preprocess_input(train_images)
X_test_img = preprocess_input(test_images)

print("Train tabular:", X_train_tab.shape, "| Train images:", X_train_img.shape)
print("Test tabular:", X_test_tab.shape, "| Test images:", X_test_img.shape)


## 4. Model Development

### Why transfer learning?
With only ~535 houses (428 for training), training a CNN from scratch would overfit badly. Instead we
use **MobileNetV2 pretrained on ImageNet** as a frozen feature extractor — it already knows general
visual features (edges, textures, shapes, materials) that transfer well to house photos, and freezing it
keeps the number of trainable parameters small relative to our dataset size.

### Architecture (feature fusion)
- **Image branch:** MobileNetV2 (frozen, `include_top=False`, global average pooling) → Dense(128, relu) → Dropout
- **Tabular branch:** Dense(16, relu) → Dense(8, relu) on the 4 scaled tabular features
- **Fusion:** Concatenate both branches → Dense(32, relu) → Dense(1, linear) to predict scaled price


In [ ]:
def build_image_branch():
    base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet", pooling="avg")
    base_model.trainable = False  # freeze pretrained weights
    x = Dense(128, activation="relu")(base_model.output)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation="relu")(x)
    return base_model.input, x

def build_tabular_branch(input_dim):
    tab_input = Input(shape=(input_dim,), name="tabular_input")
    x = Dense(16, activation="relu")(tab_input)
    x = Dense(8, activation="relu")(x)
    return tab_input, x

img_input, img_features = build_image_branch()
tab_input, tab_features = build_tabular_branch(X_train_tab.shape[1])

combined = Concatenate()([img_features, tab_features])
x = Dense(32, activation="relu")(combined)
x = Dropout(0.2)(x)
output = Dense(1, activation="linear", name="price_output")(x)

model = Model(inputs=[img_input, tab_input], outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-4), loss="mse", metrics=["mae"])
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)

history = model.fit(
    x=[X_train_img, X_train_tab],
    y=y_train,
    validation_data=([X_test_img, X_test_tab], y_test),
    epochs=60,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1,
)


## 5. Evaluation

We invert the MinMax scaling on the predictions and ground truth to get back to real dollar values
before computing **MAE** (Mean Absolute Error) and **RMSE** (Root Mean Squared Error), so the metrics
are directly interpretable as dollar amounts.


In [ ]:
y_pred_scaled = model.predict([X_test_img, X_test_tab])

# Invert scaling back to real dollar values
y_pred = price_scaler.inverse_transform(y_pred_scaled)
y_true = price_scaler.inverse_transform(y_test)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"MAPE: {mape:.2f}%")


## 6. Visualizations

In [ ]:
# Training/validation loss curves
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("MSE (scaled)"); plt.title("Loss Curve"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["mae"], label="Train MAE")
plt.plot(history.history["val_mae"], label="Val MAE")
plt.xlabel("Epoch"); plt.ylabel("MAE (scaled)"); plt.title("MAE Curve"); plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Predicted vs Actual price scatter plot
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.6, edgecolor="k")
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
plt.plot(lims, lims, "r--", label="Perfect Prediction")
plt.xlabel("Actual Price ($)")
plt.ylabel("Predicted Price ($)")
plt.title("Predicted vs Actual House Prices")
plt.legend()
plt.show()


In [ ]:
# Residuals distribution
residuals = (y_true - y_pred).flatten()
plt.figure(figsize=(6, 4))
sns.histplot(residuals, kde=True, bins=25)
plt.axvline(0, color="red", linestyle="--")
plt.xlabel("Residual (Actual - Predicted) $")
plt.title("Residual Distribution")
plt.show()


## 7. Final Summary / Insights

- We built a **multimodal fusion model** combining a frozen MobileNetV2 image branch (fed a 2x2 montage
  of each house's bathroom/bedroom/frontal/kitchen photos) with an MLP branch over tabular features
  (bedrooms, bathrooms, area, and target-encoded zipcode).
- Target-encoding zipcode (instead of one-hot encoding) kept the tabular input compact and let the model
  capture location effects on price without exploding dimensionality on a 535-row dataset.
- Transfer learning (freezing MobileNetV2's ImageNet weights) was essential given the small dataset size
  — it avoided the CNN overfitting while still contributing meaningful visual signal.
- **Fill in after running:** report the final MAE / RMSE / MAPE printed above, and note whether the
  image branch improved results versus a tabular-only baseline (you can test this by removing the image
  branch and retraining, as an ablation).
- **Possible improvements:** fine-tune the last few MobileNetV2 layers once the head has converged,
  try per-image (rather than montage) feature extraction with attention/pooling across the 4 rooms, or
  gather a larger dataset to reduce variance in the test metrics.
